In [9]:
import time
import os
import json
import pandas as pd
from bert_score import score
from src.NewsAnalyzerAPI import NewsArticle
from src.NewsAnalyzerAPI import LLMConnector
from src.NewsAnalyzerAPI import NewsAnalyzer
from tqdm import tqdm

In [10]:
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("API_KEY")

In [11]:
import logging
from transformers import logging as transformers_logging

# Configuring log level to suppress unwanted warnings
logging.basicConfig(level=logging.ERROR)
transformers_logging.set_verbosity_error()

In [12]:
con = LLMConnector("http://143.107.183.116:18888/v1", api_key, "llama3.1")
analyzer = NewsAnalyzer(con)

In [13]:
# file_path: path to the file with json object with news data
# data_list: list with news data
def extract_texts(file_path, data_list):
    # extracts specified component from json object
    def extract_component(data, component):
        annotations = data["fiveWoneH"][component]["annotated"]
        texts = [item.get("text") for item in annotations]
        return "; ".join(text for text in texts if text is not None)

    # loads json object from file
    with open(file_path, "r") as file:
        data = json.load(file)

    # print("1")
    text = data["text"]
    what_true = extract_component(data, "what")
    where_true = extract_component(data, "where")
    when_true = extract_component(data, "when")
    who_true = extract_component(data, "who")
    why_true = extract_component(data, "why")
    how_true = extract_component(data, "how")
    # print("1")

    # print(data)
    # print("2")
    article = NewsArticle(data.get("title"), data.get("description"), text, data.get("date_publish"), data.get("url"))
    # print("what")
    what_pred = analyzer.identify_component(article, "what")
    
    # print("where")
    where_pred = analyzer.identify_component(article, "where")
    
    # print("when")
    when_pred = analyzer.identify_component(article, "when")
    
    # print("who")
    who_pred = analyzer.identify_component(article, "who")
    
    # print("why")
    why_pred = analyzer.identify_component(article, "why")
    
    # print("how")
    how_pred = analyzer.identify_component(article, "how")
    # print("3")
    # print("2")

    data_list.append({
        "text": text,
        "what_true": what_true,
        "where_true": where_true,
        "when_true": when_true,
        "who_true": who_true,
        "why_true": why_true,
        "how_true": how_true,
        "what_pred": what_pred,
        "where_pred": where_pred,
        "when_pred": when_pred,
        "who_pred": who_pred,
        "why_pred": why_pred,
        "how_pred": how_pred
    })

def evaluate(data_list):
    e = []
    for article in data_list:
        cands = [article[component] for component in article if component.endswith("_pred")]
        refs = [article[component] for component in article if component.endswith("_true")]
        start = time.time()
        P, R, F1 = score(cands, refs, lang="en")
        end = time.time()
        print(F1)
        print(f"System level F1 score: {F1.mean():.3f}")
        print("Tempo: ", end-start)
        e.append([article, cands, refs, P, R, F1, end-start])

    return e

In [14]:
data_list = []
extract_texts("./data_samples/0e5fa7c0e6252bfeeea5e3840c6cb503f299c19d24331c4ba60c5974.json", data_list)
print(data_list)

[{'text': 'Skip Ad Ad Loading... x Embed x Share Toblerone is facing a mountain of criticism for changing the shape of its famous triangular candy bars in British stores, a move it blames on rising costs. USA TODAY Toblerone chocolate bars come in a variety of sizes, but recently changed the shape of two of its smaller bars sold in the UK. (Photo: Martin Ruetschi, AP) The UK has a chocolate bar crisis on its hands: the beloved Swiss chocolate bar is unrecognizable. Toblerone, the classic chocolate bar with almond-and-honey-filled triangle chunks, recently lost weight. In two sizes, the triangles shrunk, leaving wider gaps of chocolate. Toblerone can you tell me what this is all about... looks like there\'s half a bar missing! pic.twitter.com/C2VD3DjppE -- Alana Cartwright (@AlanaCartwrigh3) October 29, 2016  @HelenRyles Hi Helen, yes this is just our smaller bar. -- Toblerone (@Toblerone) October 31, 2016  The 400-gram bar was reduced to a 360-gram bar and the 170-gram was reduced to 1

In [18]:
# evaluation for a single article
evaluate(data_list)

tensor([0.8226, 0.8116, 0.8271, 0.8094, 0.8289, 0.7919])
System level F1 score: 0.815


In [20]:
# extracting components for all articles and writing in a spreadsheet
data_list = []
data_folder = './data_samples/'
cnt = 1
for filename in os.listdir(data_folder):
    print(f"\n{cnt}")
    file_path = os.path.join(data_folder, filename)
    start = time.time()
    extract_texts(file_path, data_list)
    end = time.time()
    print("Tempo: ", end-start)
    cnt += 1
df = pd.DataFrame(data_list)
df.to_excel('news.xlsx', index=False)
df.to_csv('news.csv', index=False, encoding='utf-8')


1
Tempo:  9.860365629196167

2
Tempo:  12.708157062530518

3
Tempo:  17.203187704086304

4
Tempo:  15.155169248580933

5
Tempo:  16.487067461013794

6
Tempo:  23.444464683532715

7
Tempo:  17.5146427154541

8
Tempo:  19.552385568618774

9
Tempo:  25.399742126464844

10
Tempo:  15.655762910842896

11
Tempo:  34.07911992073059

12
Tempo:  20.82047724723816

13
Tempo:  19.8611478805542

14
Tempo:  19.155587911605835

15
Tempo:  23.85418176651001

16
Tempo:  21.914833068847656

17
Tempo:  23.038374423980713

18
Tempo:  15.368513107299805

19
Tempo:  21.67674946784973

20
Tempo:  17.01675772666931

21
Tempo:  22.837105989456177

22
Tempo:  14.653177738189697

23
Tempo:  7.158463954925537

24
Tempo:  11.88298749923706

25
Tempo:  16.159224033355713

26
Tempo:  15.365145444869995

27
Tempo:  10.539376974105835

28
Tempo:  24.392463207244873

29
Tempo:  22.280375242233276

30
Tempo:  22.87253761291504

31
Tempo:  21.039329528808594

32
Tempo:  25.327558279037476

33
Tempo:  12.655776500701904

In [5]:
df = pd.read_excel("news.xlsx")
df = df.fillna("")
data_list2 = df.to_dict(orient="records")

In [6]:
e = evaluate(data_list2)
df_e = pd.DataFrame(e)
df_e.to_pickle("avaliacao.pkl")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tensor([0.8298, 0.8212, 0.8116, 0.7807, 0.0000, 0.0000])
System level F1 score: 0.541
Tempo:  97.49694514274597


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tensor([0.8265, 0.8022, 0.7789, 0.8002, 0.7476, 0.0000])
System level F1 score: 0.659
Tempo:  48.72548151016235


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

tensor([0.8452, 0.7859, 0.8620, 0.7718, 0.8328, 0.8123])
System level F1 score: 0.818
Tempo:  37.32331728935242


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tensor([0.8069, 0.8081, 0.8081, 0.7813, 0.7995, 0.7898])
System level F1 score: 0.799
Tempo:  19.13394784927368


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tensor([0.8071, 0.8249, 0.7657, 0.7879, 0.8299, 0.8027])
System level F1 score: 0.803
Tempo:  22.39340901374817


In [1]:
df_e

NameError: name 'df_e' is not defined